# Experimentos: Treinamento e Refinamento de ResNet18/ResNet50 para Detecção de Cachorros via Mapas de Ativação

Este notebook realiza o treinamento do zero e o refinamento (transfer learning) das redes ResNet18 e ResNet50 no dataset Oxford Pets, com o objetivo de obter representações úteis para detecção de objetos (cachorros) via mapas de ativação. Inclui salvamento dos melhores checkpoints e discussão sobre quais camadas refinar para maximizar a qualidade dos mapas de ativação.

## 1. Download do Dataset Oxford Pets

Utilize a célula abaixo para baixar e extrair o dataset Oxford Pets. Execute apenas uma vez.

In [ ]:
# Célula para baixar os dados. Execute apenas uma vez!
from torchvision.datasets.utils import download_and_extract_archive

def download(root):
    url_images = "https://thor.robots.ox.ac.uk/~vgg/data/pets/images.tar.gz"
    url_targets = "https://thor.robots.ox.ac.uk/~vgg/data/pets/annotations.tar.gz"
    download_and_extract_archive(url_images, root, remove_finished=False)
    download_and_extract_archive(url_targets, root, remove_finished=False)

# Exemplo de uso:
# download('../data/oxford_pets')

## 2. Importação das Bibliotecas e Utilitários

Importe as bibliotecas necessárias para o experimento, incluindo torch, torchvision, dataset.py, train.py, e utilitários para manipulação de dados e modelos.

In [ ]:
import torch
import torchvision
from torch import nn
from torchvision import models
import matplotlib.pyplot as plt
import numpy as np
import os
import sys
sys.path.append('src')
import dataset
import train

## 3. Treinamento do zero: ResNet18 no Oxford Pets

Nesta etapa, treinaremos uma ResNet18 inicializada com pesos aleatórios no dataset Oxford Pets. O melhor checkpoint será salvo para avaliação posterior.

In [ ]:
params_resnet18 = {
    "bs": 256,
    "num_epochs": 100,
    "lr": 0.01,
    "weight_decay": 1e-2,
    "resize_size": 224,
    "seed": 0
}

model_resnet18 = models.resnet18(weights=None)
model_resnet18.fc = nn.Linear(model_resnet18.fc.in_features, 2)

# Treinamento do zero
# ds_train, ds_valid, logger = train.train(model_resnet18, **params_resnet18)
# O melhor checkpoint será salvo automaticamente em ../data/checkpoints/M06/best_model_resnet18_scratch.pt

## 4. Treinamento do zero: ResNet50 no Oxford Pets

Agora, treinaremos uma ResNet50 inicializada com pesos aleatórios no dataset Oxford Pets. O melhor checkpoint será salvo para avaliação posterior.

In [ ]:
params_resnet50 = {
    "bs": 128,  # Reduzido devido ao maior uso de memória da ResNet50
    "num_epochs": 100,
    "lr": 0.01,
    "weight_decay": 1e-2,
    "resize_size": 224,
    "seed": 0
}

model_resnet50 = models.resnet50(weights=None)
model_resnet50.fc = nn.Linear(model_resnet50.fc.in_features, 2)

# Treinamento do zero
# ds_train, ds_valid, logger = train.train(model_resnet50, **params_resnet50)
# O melhor checkpoint será salvo automaticamente em ../data/checkpoints/M06/best_model_resnet50_scratch.pt

## 5. Salvamento dos melhores checkpoints dos modelos treinados do zero

Os melhores checkpoints dos modelos treinados do zero são salvos automaticamente durante o treinamento. Os arquivos são salvos em `../data/checkpoints/M06/` com nomes distintos para cada arquitetura.

## 6. Refinamento (Transfer Learning): ResNet18 pré-treinada no ImageNet

Nesta etapa, carregamos a ResNet18 com pesos do ImageNet, ajustamos a última camada para 2 classes e refinamos a rede no Oxford Pets. O melhor checkpoint será salvo para avaliação posterior.

In [ ]:
from torchvision.models import ResNet18_Weights

params_resnet18_tl = {
    "bs": 256,
    "num_epochs": 50,
    "lr": 0.001,
    "weight_decay": 1e-3,
    "resize_size": 224,
    "seed": 1
}

model_resnet18_tl = models.resnet18(weights=ResNet18_Weights.DEFAULT)
model_resnet18_tl.fc = nn.Linear(model_resnet18_tl.fc.in_features, 2)

# Exemplo: refinar apenas a última camada
for param in model_resnet18_tl.parameters():
    param.requires_grad = False
model_resnet18_tl.fc.requires_grad = True

# Para refinar mais blocos, ajuste requires_grad=True para outros parâmetros
# ds_train, ds_valid, logger = train.train(model_resnet18_tl, **params_resnet18_tl)
# O melhor checkpoint será salvo automaticamente em ../data/checkpoints/M06/best_model_resnet18_tl.pt

## 7. Refinamento (Transfer Learning): ResNet50 pré-treinada no ImageNet

Agora, carregamos a ResNet50 com pesos do ImageNet, ajustamos a última camada para 2 classes e refinamos a rede no Oxford Pets. O melhor checkpoint será salvo para avaliação posterior.

In [ ]:
from torchvision.models import ResNet50_Weights

params_resnet50_tl = {
    "bs": 128,
    "num_epochs": 50,
    "lr": 0.001,
    "weight_decay": 1e-3,
    "resize_size": 224,
    "seed": 1
}

model_resnet50_tl = models.resnet50(weights=ResNet50_Weights.DEFAULT)
model_resnet50_tl.fc = nn.Linear(model_resnet50_tl.fc.in_features, 2)

# Exemplo: refinar apenas a última camada
for param in model_resnet50_tl.parameters():
    param.requires_grad = False
model_resnet50_tl.fc.requires_grad = True

# Para refinar mais blocos, ajuste requires_grad=True para outros parâmetros
# ds_train, ds_valid, logger = train.train(model_resnet50_tl, **params_resnet50_tl)
# O melhor checkpoint será salvo automaticamente em ../data/checkpoints/M06/best_model_resnet50_tl.pt

## 8. Escolha das camadas a serem refinadas para detecção via mapa de ativação

Para maximizar a qualidade dos mapas de ativação para detecção de objetos, recomenda-se refinar não apenas a última camada, mas também os últimos blocos convolucionais (por exemplo, `layer4` na ResNet18/50). Isso permite que a rede adapte suas representações finais ao novo domínio, sem perder o conhecimento prévio do ImageNet. Abaixo, um exemplo de como liberar gradientes para os últimos blocos:

In [ ]:
# Exemplo: liberar gradientes para layer4 e fc (ResNet18/ResNet50)
for param in model_resnet18_tl.parameters():
    param.requires_grad = False
for param in model_resnet18_tl.layer4.parameters():
    param.requires_grad = True
model_resnet18_tl.fc.requires_grad = True

# Repita para model_resnet50_tl se desejar
# for param in model_resnet50_tl.parameters():
#     param.requires_grad = False
# for param in model_resnet50_tl.layer4.parameters():
#     param.requires_grad = True
# model_resnet50_tl.fc.requires_grad = True

## 9. Salvamento dos melhores checkpoints dos modelos refinados

Os melhores checkpoints dos modelos refinados (transfer learning) também são salvos automaticamente durante o treinamento, em `../data/checkpoints/M06/` com nomes distintos para cada arquitetura e estratégia de refinamento.

## 10. Resumo dos checkpoints salvos para avaliação posterior

Abaixo estão os caminhos dos principais checkpoints salvos durante os experimentos. Utilize-os para avaliação qualitativa e quantitativa em notebooks futuros:

- `../data/checkpoints/M06/best_model_resnet18_scratch.pt`  (ResNet18 treinada do zero)
- `../data/checkpoints/M06/best_model_resnet50_scratch.pt`  (ResNet50 treinada do zero)
- `../data/checkpoints/M06/best_model_resnet18_tl.pt`       (ResNet18 transfer learning, ImageNet → Oxford Pets)
- `../data/checkpoints/M06/best_model_resnet50_tl.pt`       (ResNet50 transfer learning, ImageNet → Oxford Pets)

Ajuste os nomes conforme necessário para diferentes estratégias de refinamento.